# Phase 1 — Empirical Bottleneck Decomposition
**Goal:** Formally identify where every millisecond is spent in the multimodal pipeline.

$$T_{total} = T_{vision} + T_{prefill} + T_{decode}$$

All raw measurements are produced by `phase1/run_experiments.py` and saved to `phase1/results.json`.  
This notebook loads that JSON, re-runs lightweight analyses in-notebook, and produces the Phase 1 deliverable plots.

## 1. Imports & Configuration

In [ ]:
import sys, os, json, time, warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import psutil
import urllib.request, io
from PIL import Image

warnings.filterwarnings("ignore")

# ── Project root on sys.path ───────────────────────────────────────────────────
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import mlx.core as mx
try:
    import mlx.metal as metal
    HAS_METAL_API = hasattr(metal, "get_active_memory")
except Exception:
    HAS_METAL_API = False

from mlx_vlm import load, generate

# ── Global config ──────────────────────────────────────────────────────────────
MODEL_ID        = "mlx-community/SmolVLM-Instruct-4bit"
RESOLUTIONS     = [224, 336, 512, 768, 1024]
BATCH_SIZES     = [1, 2, 4, 8]
SLA_BUDGET_MS   = 500.0
DECODE_TOKENS   = 120
RESULTS_JSON    = ROOT / "phase1" / "results.json"
PROMPT          = "Describe this image in detail."
IMAGE_URL       = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"

# SmolVLM architecture constants (verified from model card)
SMOLVLM_CFG = dict(num_layers=24, hidden_size=1152, num_heads=16, patch_size=14)
# M3 roofline constants
M3_TFLOPS          = 3.6e12   # FP16 compute
M3_BANDWIDTH_GBs   = 100e9    # memory bandwidth bytes/s

# ── Plot style ─────────────────────────────────────────────────────────────────
sns.set_theme(style="darkgrid", palette="muted", font_scale=1.15)
COLORS = sns.color_palette("muted", 6)
plt.rcParams["figure.dpi"] = 130

print(f"Root   : {ROOT}")
print(f"MLX    : {mx.__version__ if hasattr(mx,'__version__') else 'installed'}")
print(f"Device : {mx.default_device()}")
print(f"Metal API available: {HAS_METAL_API}")
print(f"Results JSON: {'EXISTS' if RESULTS_JSON.exists() else 'NOT YET — run phase1/run_experiments.py first'}")

## 2. Three-Stage Latency Decomposition — Setup

Define the `LatencyRecord` dataclass and Metal-synchronised timer utility.

$$T_{total} = T_{vision} + T_{prefill} + T_{decode}$$

In [ ]:
@dataclass
class LatencyRecord:
    resolution:      int
    t_vision_ms:     float = 0.0   # image → visual embeddings
    t_prefill_ms:    float = 0.0   # prompt + vision tokens → first token
    t_decode_ms:     float = 0.0   # mean time-between-tokens
    t_total_ms:      float = 0.0
    visual_tokens:   int   = 0
    est_v_tokens:    int   = 0
    kv_cache_mb:     float = 0.0
    fragmentation:   float = 0.0

    def __post_init__(self):
        self.t_total_ms = self.t_vision_ms + self.t_prefill_ms + self.t_decode_ms


def metal_timer_ms(fn, *args, **kwargs):
    """Call fn(*args, **kwargs), force Metal sync via mx.eval(), return (result, ms)."""
    t0 = time.perf_counter()
    result = fn(*args, **kwargs)
    mx.eval()
    return result, (time.perf_counter() - t0) * 1e3


def get_process_rss_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1_048_576


def estimate_visual_tokens(res: int, patch_size: int = 14) -> int:
    return (res // patch_size) ** 2


def kv_cache_bytes(seq_len, num_layers, hidden_size, batch=1, precision_bytes=2):
    """KV cache: Batch × SeqLen × 2 × Layers × HiddenSize × PrecisionBytes"""
    return batch * seq_len * 2 * num_layers * hidden_size * precision_bytes


print("LatencyRecord and Metal timer utilities defined ✅")

## 3. Load Results JSON
Run `phase1/run_experiments.py` once to produce this file, then all remaining cells are instant.

In [ ]:
if not RESULTS_JSON.exists():
    raise FileNotFoundError(
        f"Results not found at {RESULTS_JSON}.\n"
        "Run:  python phase1/run_experiments.py  from the project root first."
    )

with open(RESULTS_JSON) as f:
    R = json.load(f)

# ── Resolution sweep DataFrame ─────────────────────────────────────────────────
sweep_rows = []
for d in R["resolution_sweep"]:
    sweep_rows.append({
        "resolution":       d["resolution"],
        "t_vision_ms":      d["t_vision_ms"],
        "t_prefill_ms":     d["t_prefill_ms"],
        "t_decode_ms":      d["t_decode_mean_ms"],
        "t_total_ms":       d["t_total_ttft_ms"],
        "visual_tokens":    d["visual_tokens"],
        "est_v_tokens":     d["est_visual_tokens"],
        "kv_cache_mb":      d["kv_cache_mb"],
        "kv_allocated_mb":  d["kv_allocated_mb"],
        "fragmentation_pct":d["fragmentation_pct"],
        "mem_rss_mb":       d["mem_rss_mb"],
    })
df_sweep = pd.DataFrame(sweep_rows)

# ── TBT curve ─────────────────────────────────────────────────────────────────
tbt_latencies = np.array(R["tbt_curve"])

# ── Batch throughput DataFrame ────────────────────────────────────────────────
df_batch = pd.DataFrame(R["batch_throughput"])

# ── KV analysis DataFrame ────────────────────────────────────────────────────
df_kv = pd.DataFrame(R["kv_cache_analysis"])

mem_base = R["memory_baseline"]

print("Results loaded ✅")
print(f"  Model         : {R['model_id']}")
print(f"  Load time     : {mem_base['model_load_time_s']:.1f}s")
print(f"  Model Δmem    : {mem_base['model_weights_mb']:.0f} MB")
print(f"  Resolutions   : {df_sweep['resolution'].tolist()}")
print(f"  TBT samples   : {len(tbt_latencies)}")
print()
display(df_sweep[["resolution","t_vision_ms","t_prefill_ms","t_decode_ms","t_total_ms","visual_tokens","fragmentation_pct"]])

## 4. T_decode — TBT Curve over 120 Tokens
Shows KV-cache growth effect: does decode slow down as the sequence gets longer?

In [ ]:
tokens = np.arange(1, len(tbt_latencies) + 1)

# Fit linear trend to detect KV-cache bandwidth degradation
slope, intercept = np.polyfit(tokens, tbt_latencies, 1)
trend = slope * tokens + intercept

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left: per-token TBT
ax = axes[0]
ax.plot(tokens, tbt_latencies, color=COLORS[0], alpha=0.7, linewidth=1.2, label="TBT per token")
ax.plot(tokens, trend, "--", color=COLORS[3], linewidth=1.8,
        label=f"Linear trend  slope={slope*1000:.3f} µs/tok")
ax.axhline(np.mean(tbt_latencies), color="gray", linestyle=":", linewidth=1.2,
           label=f"Mean = {np.mean(tbt_latencies):.1f} ms")
ax.set_xlabel("Token index"); ax.set_ylabel("TBT (ms)")
ax.set_title("T_decode: Time-Between-Tokens (KV-cache growth)")
ax.legend(fontsize=9)

# Right: rolling mean (window=10)
window = 10
rolling = pd.Series(tbt_latencies).rolling(window).mean()
ax2 = axes[1]
ax2.plot(tokens, rolling, color=COLORS[1], linewidth=2, label=f"Rolling mean (w={window})")
ax2.fill_between(tokens,
                  pd.Series(tbt_latencies).rolling(window).min(),
                  pd.Series(tbt_latencies).rolling(window).max(),
                  alpha=0.25, color=COLORS[1])
ax2.set_xlabel("Token index"); ax2.set_ylabel("TBT (ms)")
ax2.set_title("TBT Rolling Mean (memory-bandwidth pressure)")
ax2.legend(fontsize=9)

plt.suptitle("Phase 1 — T_decode Analysis", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"\nT_decode statistics:")
print(f"  Mean  : {np.mean(tbt_latencies):.2f} ms  →  {1000/np.mean(tbt_latencies):.1f} tok/s")
print(f"  p50   : {np.percentile(tbt_latencies,50):.2f} ms")
print(f"  p90   : {np.percentile(tbt_latencies,90):.2f} ms")
print(f"  p99   : {np.percentile(tbt_latencies,99):.2f} ms")
print(f"  Trend slope: {slope*1000:+.3f} µs per additional token  ",
      "← KV-cache growth visible" if slope > 0 else "← negligible growth")

## 5. Latency Breakdown: T_total = T_vision + T_prefill + T_decode
Stacked bar chart across the resolution sweep. Red dashed line = 500ms SLA budget.

In [ ]:
res_labels = [f"{r}px" for r in df_sweep["resolution"]]
x = np.arange(len(res_labels))
width = 0.55

fig, ax = plt.subplots(figsize=(12, 5.5))

bars_v = ax.bar(x, df_sweep["t_vision_ms"],  width, label="T_vision",  color=COLORS[0])
bars_p = ax.bar(x, df_sweep["t_prefill_ms"], width, bottom=df_sweep["t_vision_ms"],
                label="T_prefill", color=COLORS[1])
bars_d = ax.bar(x, df_sweep["t_decode_ms"],  width,
                bottom=df_sweep["t_vision_ms"] + df_sweep["t_prefill_ms"],
                label="T_decode",  color=COLORS[2])

# Annotate each segment
for i, (v, p, d) in enumerate(zip(df_sweep["t_vision_ms"],
                                    df_sweep["t_prefill_ms"],
                                    df_sweep["t_decode_ms"])):
    ax.text(i, v / 2,               f"{v:.0f}", ha="center", va="center", fontsize=8.5, color="white", fontweight="bold")
    ax.text(i, v + p / 2,           f"{p:.0f}", ha="center", va="center", fontsize=8.5, color="white", fontweight="bold")
    ax.text(i, v + p + d / 2,       f"{d:.0f}", ha="center", va="center", fontsize=8.5, color="white", fontweight="bold")

ax.axhline(SLA_BUDGET_MS, color="red", linestyle="--", linewidth=1.8, label=f"SLA budget ({SLA_BUDGET_MS:.0f}ms)")
ax.set_xticks(x); ax.set_xticklabels(res_labels)
ax.set_xlabel("Input Resolution"); ax.set_ylabel("Latency (ms)")
ax.set_title("Phase 1 — Latency Decomposition: T_total = T_vision + T_prefill + T_decode", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print("\nDecomposition table:")
print(df_sweep[["resolution","t_vision_ms","t_prefill_ms","t_decode_ms","t_total_ms"]].to_string(index=False))

## 6. Token Explosion Curve
**The core Phase 1 proof:** N_v_tokens vs T_prefill.  
Theory predicts $T_{prefill} \propto N_v^2$ (attention is $O(L^2)$).  
We fit both a linear and quadratic curve and show the quadratic wins.

In [ ]:
v_tok  = df_sweep["est_v_tokens"].values.astype(float)
t_pref = df_sweep["t_prefill_ms"].values
t_vis  = df_sweep["t_vision_ms"].values
res    = df_sweep["resolution"].values

# Quadratic fit: T_prefill = a*N^2 + b*N + c
coef_q = np.polyfit(v_tok, t_pref, 2)
coef_l = np.polyfit(v_tok, t_pref, 1)
x_fit  = np.linspace(v_tok.min(), v_tok.max(), 200)
y_q    = np.polyval(coef_q, x_fit)
y_l    = np.polyval(coef_l, x_fit)

# R² for quadratic fit
ss_res = np.sum((t_pref - np.polyval(coef_q, v_tok))**2)
ss_tot = np.sum((t_pref - t_pref.mean())**2)
r2_q   = 1 - ss_res / ss_tot

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Token explosion scatter + fits
ax = axes[0]
ax.scatter(v_tok, t_pref, s=90, zorder=5, color=COLORS[0], label="Measured T_prefill")
for vt, tp, r in zip(v_tok, t_pref, res):
    ax.annotate(f"{r}px", (vt, tp), textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.plot(x_fit, y_q, "--", color=COLORS[1], linewidth=2,
        label=f"Quadratic fit  R²={r2_q:.3f}")
ax.plot(x_fit, y_l, ":", color=COLORS[3], linewidth=1.8, label="Linear fit")
# Mark the "knee" — point where quadratic overtakes linear meaningfully
knee_idx = np.argmax(np.abs(y_q - y_l) > 10)
if knee_idx < len(x_fit):
    ax.axvline(x_fit[knee_idx], color="orange", linewidth=1.5, linestyle="-.",
               label=f"'Knee' ≈ {x_fit[knee_idx]:.0f} tokens")
ax.set_xlabel("N_v_tokens (visual tokens)")
ax.set_ylabel("T_prefill (ms)")
ax.set_title("Token Explosion: N_v_tokens vs T_prefill")
ax.legend(fontsize=9)

# Right: T_vision (linear) vs T_prefill (quadratic) vs resolution
ax2 = axes[1]
ax2.plot(res, t_vis,  "o-", color=COLORS[0], linewidth=2, markersize=7, label="T_vision  (linear O(n))")
ax2.plot(res, t_pref, "s-", color=COLORS[1], linewidth=2, markersize=7, label="T_prefill (quadratic O(n²))")
ax2.set_xlabel("Resolution (px)")
ax2.set_ylabel("Latency (ms)")
ax2.set_title("T_vision vs T_prefill Scaling with Resolution")
ax2.legend(fontsize=9)

plt.suptitle("Phase 1 — Token Explosion Curve", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"\nQuadratic fit: T_prefill = {coef_q[0]:.5f}·N² + {coef_q[1]:.4f}·N + {coef_q[2]:.2f}")
print(f"R² (quadratic) = {r2_q:.4f}  ← {'✅ confirms O(n²) scaling' if r2_q > 0.9 else '⚠️ data may be too flat to confirm'}")
print(f"\nVisual token counts:")
for r, vt in zip(res, v_tok):
    print(f"  {r}px → {vt:.0f} tokens  (patch_size={SMOLVLM_CFG['patch_size']})")

## 7. KV-Cache Size & Fragmentation Audit
Formula: $KV_{bytes} = Batch \times SeqLen \times 2 \times Layers \times HiddenSize \times PrecisionBytes$

Without PagedAttention, memory is pre-allocated to `max_seq_len=2048` regardless of actual usage → 60–80% waste.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: KV size vs seq_len (theoretical)
ax = axes[0]
seq_lens = df_kv["seq_len"].values
kv_mb_actual = df_kv["kv_mb"].values
kv_mb_alloc  = df_kv["allocated_mb"].values
frag         = df_kv["fragmentation_pct"].values

ax.fill_between(seq_lens, kv_mb_alloc, alpha=0.25, color=COLORS[3], label="Allocated (max_seq=2048)")
ax.plot(seq_lens, kv_mb_actual, "o-", color=COLORS[0], linewidth=2, markersize=7, label="Actual KV cache used")
ax.plot(seq_lens, kv_mb_alloc,  "--", color=COLORS[3], linewidth=1.8, label="Max allocation")
for sl, km in zip(seq_lens, kv_mb_actual):
    ax.annotate(f"{km:.2f}MB", (sl, km), textcoords="offset points", xytext=(3, 6), fontsize=8)
ax.set_xlabel("Sequence Length (tokens)")
ax.set_ylabel("KV Cache (MB)")
ax.set_title("KV-Cache Size vs Sequence Length")
ax.legend(fontsize=9)

# Right: Fragmentation % vs seq_len
ax2 = axes[1]
bars = ax2.bar(seq_lens.astype(str), frag, color=[COLORS[2] if f < 60 else COLORS[4] for f in frag])
ax2.axhline(60, color="orange", linestyle="--", linewidth=1.5, label="60% waste threshold")
ax2.axhline(80, color="red",    linestyle="--", linewidth=1.5, label="80% waste threshold")
for bar, f in zip(bars, frag):
    ax2.text(bar.get_x() + bar.get_width()/2, f + 1, f"{f:.0f}%",
             ha="center", va="bottom", fontsize=9, fontweight="bold")
ax2.set_xlabel("Sequence Length (tokens)")
ax2.set_ylabel("Fragmentation (%)")
ax2.set_title("Memory Fragmentation Audit\n(Allocated - Used) / Allocated × 100")
ax2.legend(fontsize=9)

plt.suptitle("Phase 1 — KV-Cache Analysis & Fragmentation", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Per-resolution fragmentation from sweep
print("\nFragmentation per resolution (from sweep):")
print(df_sweep[["resolution","kv_cache_mb","kv_allocated_mb","fragmentation_pct"]].to_string(index=False))

avg_frag = df_sweep["fragmentation_pct"].mean()
print(f"\n⚠️  Average fragmentation across sweep: {avg_frag:.1f}%")
print("→ Justifies Phase 4 PagedAttention KV manager" if avg_frag > 40 else "→ Fragmentation within acceptable range")

## 8. Weight vs KV-Cache Memory Ratio
Compares model weight memory (~static) vs KV-cache memory (~dynamic, grows with context).

In [ ]:
weight_mb   = mem_base["model_weights_mb"]
system_mb   = mem_base["rss_after_load_mb"]

# KV cache at max resolution (1024px, seq≈5480)
max_sweep   = df_sweep.loc[df_sweep["resolution"].idxmax()]
kv_max_mb   = max_sweep["kv_allocated_mb"]  # worst-case allocated

other_mb    = max(0, system_mb - weight_mb - kv_max_mb)
budget_mb   = 8 * 1024  # 8GB unified memory

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: stacked bar — actual vs budget
categories  = ["At Max Resolution (1024px)"]
bottom = 0
bars_data = [("Model Weights", weight_mb, COLORS[0]),
             ("KV Cache (allocated)", kv_max_mb, COLORS[1]),
             ("OS + Other", other_mb, COLORS[2])]
ax = axes[0]
for label, val, color in bars_data:
    ax.bar(categories, [val], bottom=[bottom], label=f"{label}: {val:.0f} MB", color=color, width=0.4)
    ax.text(0, bottom + val/2, f"{val:.0f} MB", ha="center", va="center",
            fontsize=10, fontweight="bold", color="white")
    bottom += val
ax.axhline(budget_mb, color="red", linestyle="--", linewidth=2, label=f"8GB budget ({budget_mb:.0f} MB)")
ax.set_ylabel("Memory (MB)")
ax.set_title("Memory Budget Breakdown at Max Resolution")
ax.set_ylim(0, budget_mb * 1.1)
ax.legend(fontsize=9, loc="upper left")

# Right: ratio bar — weight / kv across resolutions
ratios = df_sweep["kv_cache_mb"].apply(lambda kv: weight_mb / max(kv, 0.001))
ax2 = axes[1]
ax2.bar([f"{r}px" for r in df_sweep["resolution"]], ratios, color=COLORS[0], alpha=0.8)
ax2.axhline(1.0, color="gray", linestyle=":", linewidth=1.5, label="Weights = KV (ratio=1)")
ax2.set_xlabel("Resolution"); ax2.set_ylabel("Weight MB / KV Cache MB")
ax2.set_title("Weight-to-KV Ratio\n(higher → weights dominate)")
ax2.legend()
for i, (r, ratio) in enumerate(zip(df_sweep["resolution"], ratios)):
    ax2.text(i, ratio + 0.05, f"{ratio:.1f}×", ha="center", fontsize=9)

plt.suptitle("Phase 1 — Weight vs KV-Cache Memory", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"\nModel weights   : {weight_mb:.0f} MB  ({weight_mb/1024:.2f} GB)")
print(f"KV cache (max)  : {kv_max_mb:.1f} MB  (worst-case allocated)")
print(f"RSS after load  : {system_mb:.0f} MB  ({system_mb/1024:.2f} GB)")
print(f"8GB budget      : {budget_mb:.0f} MB")
print(f"Headroom        : {budget_mb - system_mb:.0f} MB")

## 9. Batch Throughput & SLA Violation Pareto Frontier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bs      = df_batch["batch_size"].values
tps     = df_batch["tokens_per_sec"].values
ftms    = df_batch["first_token_ms"].values
sla_ok  = df_batch["sla_pass"].values

# Left: Pareto frontier — throughput vs first-token latency
ax = axes[0]
scatter = ax.scatter(ftms, tps,
                     c=["green" if ok else "red" for ok in sla_ok],
                     s=130, zorder=5, edgecolors="white", linewidth=1.5)
for b, ft, tp in zip(bs, ftms, tps):
    ax.annotate(f"B={b}", (ft, tp), textcoords="offset points", xytext=(7, 4), fontsize=10)
ax.axvline(SLA_BUDGET_MS, color="red", linestyle="--", linewidth=1.8,
           label=f"SLA budget ({SLA_BUDGET_MS:.0f}ms)")
green_patch = mpatches.Patch(color="green", label="SLA PASS")
red_patch   = mpatches.Patch(color="red",   label="SLA FAIL")
ax.legend(handles=[green_patch, red_patch,
                   plt.Line2D([0],[0], color="red", linestyle="--", label=f"SLA {SLA_BUDGET_MS:.0f}ms")],
          fontsize=9)
ax.set_xlabel("First Token Latency (ms)")
ax.set_ylabel("Throughput (tok/s)")
ax.set_title("Pareto Frontier: Throughput vs Latency")

# Right: throughput bar chart
ax2 = axes[1]
bar_colors = ["green" if ok else "red" for ok in sla_ok]
bars2 = ax2.bar([f"B={b}" for b in bs], tps, color=bar_colors, alpha=0.85)
for bar, tp, ok in zip(bars2, tps, sla_ok):
    ax2.text(bar.get_x() + bar.get_width()/2, tp + 0.5,
             f"{tp:.1f}\n{'✅' if ok else '❌'}", ha="center", fontsize=10, fontweight="bold")
ax2.set_xlabel("Batch Size"); ax2.set_ylabel("Throughput (tok/s)")
ax2.set_title("Throughput by Batch Size\n(green=SLA pass, red=SLA fail)")

plt.suptitle("Phase 1 — Throughput vs Latency Trade-off", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\nBatch throughput results:")
print(df_batch[["batch_size","tokens_per_sec","first_token_ms","sla_pass"]].to_string(index=False))

violations = (~sla_ok).sum()
print(f"\nSLA violations: {violations}/{len(sla_ok)} batch configurations exceed {SLA_BUDGET_MS:.0f}ms")

## 10. SLA Violation Heatmap: Resolution × Batch Size

Combine resolution sweep TTFT data with batch-size latency overhead to produce a grid of estimated violation rates.

In [ ]:
# Build a synthetic but data-grounded heatmap:
# Estimated TTFT(res, batch) = T_total(res) * batch_latency_multiplier(batch)
# batch_multiplier = first_token_ms(batch) / first_token_ms(batch=1)
base_ft = df_batch.loc[df_batch["batch_size"] == 1, "first_token_ms"].values
base_ft = float(base_ft[0]) if len(base_ft) else 1.0
batch_multipliers = {
    int(row["batch_size"]): row["first_token_ms"] / base_ft
    for _, row in df_batch.iterrows()
}

grid_rows = []
for _, sr in df_sweep.iterrows():
    row = {}
    for bs_val, mult in batch_multipliers.items():
        est_ttft = sr["t_total_ms"] * mult
        # Violation = 1 if TTFT > SLA
        row[f"B={bs_val}"] = min(100.0, (est_ttft / SLA_BUDGET_MS) * 100)
    grid_rows.append(row)

df_heat = pd.DataFrame(grid_rows,
                        index=[f"{r}px" for r in df_sweep["resolution"]])

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(df_heat, annot=True, fmt=".0f", cmap="RdYlGn_r",
            vmin=0, vmax=200, ax=ax,
            cbar_kws={"label": "% of SLA budget consumed (100% = 500ms)"})
ax.set_xlabel("Batch Size")
ax.set_ylabel("Resolution")
ax.set_title("SLA Budget Consumption Heatmap\n(>100% = SLA violation)", fontweight="bold")
plt.tight_layout()
plt.show()

violations_grid = (df_heat > 100).sum().sum()
total_cells = df_heat.size
print(f"\nSLA violations in grid: {violations_grid}/{total_cells} configurations exceed {SLA_BUDGET_MS:.0f}ms budget")
print("\nFull heatmap values (% of budget consumed):")
print(df_heat.round(1).to_string())

## 11. Hardware Utilisation Map — Compute-Bound vs Memory-Bound

Use the **Roofline Model** with M3 specs:
- Peak compute: **3.6 TFLOPS** (FP16)  
- Memory bandwidth: **100 GB/s**  
- Ridge point (arithmetic intensity): $3.6\text{T} / 100\text{G} = 36\ \text{FLOP/byte}$

If a stage's arithmetic intensity > 36 FLOP/byte → **compute-bound**.  
If < 36 → **memory-bound**.

In [ ]:
ridge_point = M3_TFLOPS / M3_BANDWIDTH_GBs   # FLOP/byte

def arithmetic_intensity_prefill(n_vtok, hidden, num_heads):
    """
    Prefill: attention over L tokens.
    FLOPs ≈ 2 * L^2 * H  (QK^T and AV)  +  2 * L * H * 4H  (FFN)
    Bytes ≈ L * H * 2  (read activations FP16)  +  weight bytes
    """
    L = n_vtok
    H = hidden
    flops = 2 * L**2 * H + 2 * L * H * 4 * H
    bytes_moved = L * H * 2 + 4 * H**2 * 2   # activations + weights (one layer, FP16)
    return flops / bytes_moved

def arithmetic_intensity_decode(hidden):
    """
    Decode: single token, reads full KV cache + weights.
    FLOPs ≈ 2 * H * 4H  (one token through FFN)
    Bytes ≈ KV cache size + weight row  (memory-bandwidth dominated)
    """
    H = hidden
    flops = 2 * H * 4 * H          # single token FFN
    bytes_moved = 2 * H * 2 + H * 4 * H * 2  # KV row + weight
    return flops / bytes_moved

rows = []
for _, sr in df_sweep.iterrows():
    res = int(sr["resolution"])
    N   = int(sr["est_v_tokens"])
    H   = SMOLVLM_CFG["hidden_size"]
    nh  = SMOLVLM_CFG["num_heads"]

    ai_p = arithmetic_intensity_prefill(N, H, nh)
    ai_d = arithmetic_intensity_decode(H)

    prefill_roofline = min(M3_TFLOPS, ai_p * M3_BANDWIDTH_GBs)
    decode_roofline  = min(M3_TFLOPS, ai_d * M3_BANDWIDTH_GBs)

    rows.append({
        "resolution":         res,
        "N_v_tokens":         N,
        "AI_prefill":         round(ai_p, 1),
        "AI_decode":          round(ai_d, 3),
        "prefill_bound":      "🔴 Compute" if ai_p > ridge_point else "🔵 Memory",
        "decode_bound":       "🔴 Compute" if ai_d > ridge_point else "🔵 Memory",
        "prefill_roof_TFLOPS":round(prefill_roofline / 1e12, 3),
        "decode_roof_TFLOPS": round(decode_roofline  / 1e12, 3),
    })

df_roofline = pd.DataFrame(rows)

# Plot roofline
fig, ax = plt.subplots(figsize=(10, 5.5))
ai_range = np.logspace(-2, 3, 300)
roofline_perf = np.minimum(M3_TFLOPS / 1e12, ai_range * M3_BANDWIDTH_GBs / 1e12)
ax.loglog(ai_range, roofline_perf, "k-", linewidth=2.5, label="M3 Roofline")
ax.axvline(ridge_point, color="gray", linestyle="--", linewidth=1.5,
           label=f"Ridge point ({ridge_point:.0f} FLOP/byte)")

# Plot each resolution's prefill and decode
for _, row in df_roofline.iterrows():
    ai_p  = row["AI_prefill"]
    ai_d  = row["AI_decode"]
    perf_p = min(M3_TFLOPS / 1e12, ai_p * M3_BANDWIDTH_GBs / 1e12)
    perf_d = min(M3_TFLOPS / 1e12, ai_d * M3_BANDWIDTH_GBs / 1e12)
    ax.scatter(ai_p, perf_p, s=110, color=COLORS[1], zorder=5)
    ax.scatter(ai_d, perf_d, s=110, color=COLORS[2], marker="s", zorder=5)
    ax.annotate(f"{row['resolution']}px\nprefill", (ai_p, perf_p),
                textcoords="offset points", xytext=(6, 4), fontsize=7.5, color=COLORS[1])

# Legend proxies
ax.scatter([], [], color=COLORS[1], s=80, label="Prefill stage")
ax.scatter([], [], color=COLORS[2], s=80, marker="s", label="Decode stage (all res.)")
ax.set_xlabel("Arithmetic Intensity (FLOP / byte)")
ax.set_ylabel("Performance (TFLOPS)")
ax.set_title("Roofline Model — M3 Mac (3.6 TFLOPS, 100 GB/s)\nPrefill vs Decode Binding", fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nM3 Ridge point: {ridge_point:.0f} FLOP/byte\n")
print(df_roofline[["resolution","N_v_tokens","AI_prefill","prefill_bound","AI_decode","decode_bound"]].to_string(index=False))

print("\n━━━ Hardware Utilisation Summary ━━━")
print(f"  Prefill stage: {df_roofline['prefill_bound'].iloc[-1]} bound at max resolution")
print(f"  Decode stage : {df_roofline['decode_bound'].iloc[0]} bound (all resolutions)")
print("\n→ Decode is always memory-bandwidth bound on M3.")
print("→ Prefill transitions to compute-bound at higher resolutions (more tokens, denser attention).")

## 12. Phase 1 Summary & Implications for Phase 2

In [ ]:
best_res     = df_sweep.loc[df_sweep["t_total_ms"] <= SLA_BUDGET_MS, "resolution"].max() \
               if (df_sweep["t_total_ms"] <= SLA_BUDGET_MS).any() else "none"
worst_frag   = df_sweep["fragmentation_pct"].max()
mean_tbt     = np.mean(tbt_latencies)
r2_note      = f"R²={r2_q:.3f}" if 'r2_q' in dir() else ""

print("=" * 70)
print("PHASE 1 — EMPIRICAL BOTTLENECK DECOMPOSITION SUMMARY")
print("=" * 70)

print(f"""
Model          : {R['model_id']}
Hardware       : Apple M3, 8GB unified memory

━━━ Three-Stage Latency (336px baseline) ━━━
  T_vision   : {df_sweep.loc[df_sweep['resolution']==336, 't_vision_ms'].values[0]:.1f} ms  (image → visual embeddings)
  T_prefill  : {df_sweep.loc[df_sweep['resolution']==336, 't_prefill_ms'].values[0]:.1f} ms  (LLM prompt + vision tokens → first token)
  T_decode   : {mean_tbt:.1f} ms/token  ({1000/mean_tbt:.1f} tok/s)
  T_total    : {df_sweep.loc[df_sweep['resolution']==336, 't_total_ms'].values[0]:.1f} ms  (TTFT at 336px)

━━━ Token Explosion {r2_note} ━━━
  N_v_tokens grows as (res / {SMOLVLM_CFG['patch_size']})² → T_prefill scales quadratically.
  Largest resolution tested: {df_sweep['resolution'].max()}px → {int(df_sweep['est_v_tokens'].max())} visual tokens.

━━━ Memory / Fragmentation ━━━
  Model weights   : {mem_base['model_weights_mb']:.0f} MB
  KV fragmentation: up to {worst_frag:.0f}% waste without PagedAttention  ← Phase 4 target
  System RSS      : {mem_base['rss_after_load_mb']:.0f} MB / 8192 MB budget

━━━ SLA (500ms budget) ━━━
  Max resolution within SLA: {best_res}px
  Batch violations: {(~df_batch['sla_pass']).sum()}/{len(df_batch)} batch configs exceed budget

━━━ Hardware Bound ━━━
  Prefill : compute-bound at high resolution (attention O(L²) dominates)
  Decode  : memory-bandwidth bound (KV-cache reads dominate, 100 GB/s ceiling)

━━━ Phase 2 Cost Model Inputs ━━━
  • T_vision(res) ≈ linear with pixel count
  • T_prefill(N)  ≈ quadratic with N_v_tokens  {r2_note}
  • T_decode      ≈ {mean_tbt:.1f}ms + {slope*1000:.3f}µs × token_index  (KV growth)
  • KV(seq_len)   = {SMOLVLM_CFG['num_layers']} × {SMOLVLM_CFG['hidden_size']} × 2 × seq_len × 2 bytes
""")
print("=" * 70)